# Block 3 — Deep Learning with Time Series Data: Modern Architectures

**Goals for this block:**
- Understand differences between LSTM, GRU and TCN.
- Connect receptive field (TCN) and context window (Transformer/RNN) to the sliding-window setup.
- Run a forecasting with TCN and (optional: Transformer/ARIMA/Prophet).

## 0. Setup & Environment

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=6500)])

import torch
assert torch.cuda.is_available()
torch.cuda.set_per_process_memory_fraction(0.3)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

from darts.dataprocessing.transformers import Scaler
from darts.utils.missing_values import fill_missing_values


the preprocessing function from previous blocks

In [ ]:
def preprocessing_data_with_DARTS(series,
                      resample_freq='h', agg_method='mean',
                      missing_strategy='linear', 
                      scaling_method='standard',
                      train_ratio=0.7, val_ratio=0.15):
    """
    Complete time series preprocessing pipeline using Darts library.
        
    Returns: train, train_y, val, val_y, test, test_y, scaler
    """

    # 1. Handle missing values
    filled_series = fill_missing_values(series, method=missing_strategy)

    # 2. Resample data
    resampled_series = filled_series.resample(resample_freq, method=agg_method)
    
    # 3. Time-aware split
    
    # Calculate split points
    total_len = len(resampled_series)
    train_end = int(total_len * train_ratio)
    val_end = int(total_len * (train_ratio + val_ratio))
    
    # Split the series
    train_series = resampled_series[:train_end]
    val_series = resampled_series[train_end:val_end]
    test_series = resampled_series[val_end:]

    # 4. Scaling (fit only on training data)
    if scaling_method == 'standard':
        scaler = StandardScaler()
    elif scaling_method == 'minmax':
        scaler = MinMaxScaler()
    elif scaling_method == 'robust':
        scaler = RobustScaler()
    else:
        raise ValueError(f"Unsupported scaling method: {scaling_method}")
        
    scaler = Scaler(scaler)
    
    # Fit on training data and transform all sets
    scaler.fit(train_series)
    train_scaled = scaler.transform(train_series)
    val_scaled = scaler.transform(val_series)
    test_scaled = scaler.transform(test_series)
    
    # 5. WINDOWING
    ######## NOT NEEDED WITH DARTS MODELS ##########

    return train_scaled, val_scaled, test_scaled, scaler

## Loading Elctricity Consumption Dataset

Electricity Consumption of **households & SMEs (low voltage)** and **businesses & services (medium voltage)** in the city of Zurich, with values recorded every **15 minutes**.

The electricity consumption is combined with weather measurements recorded by **three different stations in the city of Zurich** with a hourly frequency . 

Both dataset sources are updated continuously, but this dataset only retrains values between **2015-01-01** and **2022-08-31**. The time index was converted from CET time zone to UTC.

Components Descriptions:

`Value_NE5` : Households & SMEs electricity consumption (low voltage, grid level 7) in kWh

`Value_NE7` : Business and services electricity consumption (medium voltage, grid level 5) in kWh

`Hr [%Hr]` : Relative humidity

`RainDur [min]` : Duration of precipitation (divided by 4 for conversion from hourly to quarter-hourly records)

`T [°C]` : Temperature

`WD [°]` : Wind direction

`WVv [m/s]` : Wind vector speed

`p [hPa]` : Air pressure

`WVs [m/s]` : Wind scalar speed

`StrGlo [W/m2]` : Global solar irradiation

In [ ]:
from darts.datasets import ElectricityConsumptionZurichDataset

series = ElectricityConsumptionZurichDataset().load() 
target_columns = ["Value_NE5", "Value_NE7"]
covariate_columns = ['Hr [%Hr]', 'RainDur [min]', 'StrGlo [W/m2]', 'T [°C]', 'WD [°]', 'WVs [m/s]', 'WVv [m/s]', 'p [hPa]']

In [ ]:
series.to_dataframe().head()

In [ ]:
# Plot actual values on first subplot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
series["Value_NE5"][:5000].plot(label ="Households (low voltage)", color='blue', ax=ax1)
series["Value_NE7"][:5000].plot(label="Business (medium voltage)", color='red', title= "Electricity Consumption Zurich ", ax=ax1)
ax1.set_ylabel("Electricity Consumption (MWh)")
series[covariate_columns][:5000].plot( ax=ax2)
ax2.set_ylabel("Covariate Values")

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Preprocess the target and covariate time-series</h2>

1. split the time series into two different series
   `target_series`and `past_covariates`
2. try out different frequencies (half-hourly, hourly, daily). Keep in mind we are training deep learning models (data hungry).
3. preprocess both series using the exact same parameters
4. add `future_covariates`using the `datetime_attribute_timeseries`function
   - add the **hour** attribute 
   - add the **day** attribute
   - add the **month** attribute

This exercise will help you understand how to deal with covariates.
</div>

In [ ]:
# splt the time series into target and covariates
target_series = ...
past_covariates = ...

In [ ]:
train_target, val_target, test_target, scaler_target = preprocessing_data_with_DARTS(target_series,
                                              ...)

train_past_cov, val_past_cov, test_past_cov, scaler_past_cov = preprocessing_data_with_DARTS(past_covariates,
                                              ...)
                                                                                                    

# Adding future covariates to the data

Future covariates are **known or predictable covariates** that extend into the future (e.g., holidays, planned events, calendar features).

`datetime_attribute_timeseries` creates time-based features from your time series index. These features help models understand temporal patterns like:
- **Hour of day** (0-23)
- **Day of week** (0-6)
- **Month** (1-12)

### Example:
```python
# Original timestamp: 2021-07-25 14:30:00 (Sunday afternoon)

# Hour extraction (cyclic encoding)
hour_series = datetime_attribute_timeseries(
    series, attribute="hour", one_hot=False, cyclic=True
)
```
### Parameters Explained:
- `attribute`: Which time component to extract (`"hour"`, `"day_of_week"`, `"month"`)
- `one_hot=True`: Creates binary columns (e.g., `hour_0`, `hour_1`, ..., `hour_23`)
- `one_hot=False`: Creates a single numeric column
- `cyclic=True`: Ensures continuity (hour 23 → hour 0, December → January)


In [ ]:
# adding future covariates to the data
from darts.utils.timeseries_generation import datetime_attribute_timeseries


preprocessed_series = train_target.append(val_target).append(test_target)

# create future covariates
hour_series  = datetime_attribute_timeseries(...)
dow_series   = datetime_attribute_timeseries(...)
month_series = datetime_attribute_timeseries(...)

future_covs  = hour_series.stack(dow_series).stack(month_series)

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Train an RNN model with and without covariates</h2>

1. Train an **RNN model** only on target series

2. Train an **RNN model** with the past and future covariate series
    Use both **past covariates** (weather data) and **future covariates** (time features)

3. Evaluate and compare the models using MAE, RMSE, MAPE and sMAPE metrics

6. Visualize the predictions vs actual values

</div>


In [ ]:
from darts.models import BlockRNNModel

#### RNN model using only target series

In [ ]:
window_size = 24*3  # 3 days
forecast_horizon = 24  # one day
batch_size = 256
hidden_size = 64
n_rnn_layers = 2

model_simple = BlockRNNModel(model="RNN",
                input_chunk_length=window_size,
                hidden_dim=hidden_size, 
                n_rnn_layers=n_rnn_layers,
                batch_size=batch_size,
                output_chunk_length=forecast_horizon,
                random_state=42,
                pl_trainer_kwargs={"accelerator": "gpu"})

HINT: To calculate the training time use `time.time()` to get the python time.

In [ ]:
# Train the model using only target series. Set the epochs to 2. calculate the training time.
start_time = time.time()

model_simple.fit(
    ...)

end_time = time.time()
training_time_rnn_simple = end_time - start_time

In [ ]:
# Make predictions on the test set (use the historical_forecasts method)
 
pred_series_rnn_simple = model_simple.historical_forecasts(...)

# Inverse transform the predictions and actuals
pred_series_rnn_simple_unscaled = ...
test_series_unscaled = ...

In [ ]:
from darts.metrics import mae, rmse, mape, smape
# calculate evaluation metrics

evaluation_metrics_simple_rnn = {
    "MAE": ...,
    "RMSE": ...,
    "MAPE": ...,
    "sMAPE": ...,
}

#### RNN model using past and future covariates

In [ ]:
# keep the same parameters as the simple RNN 
model_cov = BlockRNNModel(model="RNN",
                input_chunk_length=window_size,
                hidden_dim=hidden_size, 
                n_rnn_layers=n_rnn_layers,
                batch_size=batch_size,
                output_chunk_length=forecast_horizon,
                random_state=42,
                pl_trainer_kwargs={"accelerator": "gpu"})

In [ ]:
# Train the model using only target series. Set the epochs to 2. calculate the training time.
start_time = ...

model_cov.fit(
    ...)

end_time = ...
training_time_rnn_cov = ...

In [ ]:
# Make predictions on the test set (use the historical_forecasts method)

pred_series_rnn_cov = model_cov.historical_forecasts(...)

# Inverse transform the predictions and actuals
pred_series_rnn_cov_unscaled = ...
test_series_unscaled = ...

In [ ]:
# calculate evaluation metrics

evaluation_metrics_cov_rnn = {
    "MAE": ...,
    "RMSE": ...,
    "MAPE": ...,
    "sMAPE": ...,
}

In [ ]:
print("RNN with past and future covariates:", evaluation_metrics_cov_rnn)
print("\n")
print("Training time RNN with covariates (seconds):", training_time_rnn_cov)
print("\n")
print("################################")
print("\n") 
print("RNN without covariates:", evaluation_metrics_simple_rnn)
print("\n")
print("Training time RNN without covariates (seconds):", training_time_rnn_simple)

In [ ]:
# Plotting some predictions vs actuals
start_idx = pd.Timestamp('2021-07-25 03:00:00')
end_idx = pd.Timestamp('2021-08-02 03:00:00')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
test_series_unscaled[target_columns[0]][start_idx:end_idx].plot(label="Actual", color='blue', ax=ax1)
pred_series_rnn_cov_unscaled[target_columns[0]][start_idx:end_idx].plot(label="Forecast Cov RNN", color='green', ax=ax1)
pred_series_rnn_simple_unscaled[target_columns[0]][start_idx:end_idx].plot(label="Forecast Simple RNN", color='red', title=f"Actual vs Predicted Energy Consumption {target_columns[0]}", ax=ax1)

test_series_unscaled[target_columns[1]][start_idx:end_idx].plot(label="Actual", color='blue', ax=ax2)
pred_series_rnn_cov_unscaled[target_columns[1]][start_idx:end_idx].plot(label="Forecast Cov RNN", color='green', ax=ax2)
pred_series_rnn_simple_unscaled[target_columns[1]][start_idx:end_idx].plot(label="Forecast Simple RNN", color='red', title=f"Actual vs Predicted Energy Consumption {target_columns[1]}", ax=ax2)

ax1.legend()
ax2.legend()
plt.tight_layout()
plt.show()

Now we will explore the implementation of modern deep learning architectures for time series forecasting using the Darts library. In this section, we'll compare three powerful approaches:

### **Variants of Recurrent Neural Networks (RNN)**
- **GRU (Gated Recurrent Unit)**: A simplified LSTM variant that captures temporal dependencies
- **LSTM (Long Short-Term Memory)**: Classic architecture for sequential data with memory cells

### **Temporal Convolutional Networks (TCN)**
- **Key advantage**: Parallelizable training (unlike RNNs)
- **Receptive field**: Controlled by kernel size and dilation patterns
- **Best for**: Local patterns and seasonal dependencies

### **Transformer Models**
- **Attention mechanism**: Captures long-range dependencies efficiently
- **Context window**: Direct access to entire input sequence
- **Best for**: Complex temporal relationships and non-local patterns

Let's implement and compare these architectures:

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Train a LSTM, GRU, TCN models</h2>

1. Train an **LSTM model** using the preprocessed electricity consumption data

(Use both **past covariates** (weather data) and **future covariates** (time features))

2. Train a **GRU model** with the same parameters for comparison

(Use both **past covariates** (weather data) and **future covariates** (time features))

3. Train a **TCN model** and compare performance

(Use **past covariates** (weather data). Attention: TCN does not handle future covariates)

4. Evaluate and compare the models using sMAPE metrics

5. Visualize the predictions vs actual values

</div>


### RNN model

In [ ]:
window_size = 24*3  # 3 days
forecast_horizon = 24  # one day
batch_size = 256
hidden_size = 64
n_rnn_layers = 2

In [ ]:
# get the results variable from the last RNN model with covariates
model_rnn = model_cov
pred_series_rnn_unscaled = pred_series_rnn_cov_unscaled
evaluation_metrics_rnn = evaluation_metrics_cov_rnn
training_time_rnn = training_time_rnn_cov

### LTSM model

In [ ]:
# LSTM model
model_lstm = ...

# Train the model using only target series. Set the epochs to 2. calculate the training time.


# Make predictions on the test set (use the historical_forecasts method)
pred_series_lstm = ...

# Inverse transform the predictions and actuals
pred_series_lstm_unscaled = ...

# calculate evaluation metrics
evaluation_metrics_lstm = {
    "MAE": ...,
    "RMSE": ...,
    "MAPE": ...,
    "sMAPE": ...,
}

### GRU

In [ ]:
# GRU model
model_gru = ...

...

### TCN

#### TCN Model Architecture:

- Temporal Convolutional Network with dilated causal convolutions

- Parallelizable training (unlike RNNs)

- Receptive field controlled by kernel size and dilation patterns

- Residual connections and dropout for regularization

**Example Implementation:**

```python
from darts.models import TCNModel

model_tcn = TCNModel(
    input_chunk_length=window_size,     # 168 timesteps lookback (1 week)
    output_chunk_length=forecast_horizon, # 24 timesteps forecast (1 day)
    kernel_size=5,                      # Convolutional kernel size
    num_filters=64,                     # Number of filters per layer
    num_layers=3,                       # Number of TCN layers
    dilation_base=2,                    # Exponential dilation factor
    batch_size=256,                     # Batch size for training
    pl_trainer_kwargs={"accelerator": "gpu"},
    random_state=42,
)
```
**What’s happening**:
kernel_size=5: Size of the 1D convolutional kernel. Controls local pattern detection.

💡 Larger kernels capture longer local dependencies but increase parameters.

num_filters=64: Number of convolutional filters per layer (width of the network).

💡 More filters → more capacity to learn diverse temporal patterns.

num_layers=3: Depth of the TCN (number of dilated conv blocks stacked).

💡 More layers increase the receptive field exponentially with dilations.

dilation_base=2: Exponential growth factor for dilations (1, 2, 4, 8, ...).

💡 Higher base → faster growth of receptive field but potential gaps in coverage.

In [ ]:
from darts.models import TCNModel  

In [ ]:
# TCN model
...

In [ ]:
print("RNN:", evaluation_metrics_rnn)
print("Training time RNN(seconds):", training_time_rnn)
print("\n")
print("################################")
print("\n") 
print("LSTM:", evaluation_metrics_lstm)
print("Training time LSTM (seconds):", training_time_lstm)
print("\n")
print("################################")
print("\n") 
print("GRU:", evaluation_metrics_gru)
print("Training time GRU (seconds):", training_time_gru)
print("\n")
print("################################")
print("\n") 
print("TCN:", evaluation_metrics_tcn)
print("Training time TCN (seconds):", training_time_tcn)    

In [ ]:
# plotting some predictions vs actuals
start_idx = pd.Timestamp('2021-07-25 03:00:00')
end_idx = pd.Timestamp('2021-08-02 03:00:00')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
test_series_unscaled[target_columns[0]][start_idx:end_idx].plot(label="Actual", color='blue', ax=ax1)
pred_series_rnn_unscaled[target_columns[0]][start_idx:end_idx].plot(label=" RNN", color='green', ax=ax1, alpha=0.7)
pred_series_lstm_unscaled[target_columns[0]][start_idx:end_idx].plot(label=" LSTM", color='red', ax=ax1, alpha=0.7)
pred_series_gru_unscaled[target_columns[0]][start_idx:end_idx].plot(label=" GRU", color='orange', ax=ax1, alpha=0.7)
pred_series_tcn_unscaled[target_columns[0]][start_idx:end_idx].plot(label=" TCN", color='purple', alpha=0.7, title=f"Actual vs Predicted Energy Consumption {target_columns[0]}", ax=ax1)

test_series_unscaled[target_columns[1]][start_idx:end_idx].plot(label="Actual", color='blue', ax=ax2)
pred_series_rnn_unscaled[target_columns[1]][start_idx:end_idx].plot(label=" RNN", color='green', ax=ax2, alpha=0.7)
pred_series_lstm_unscaled[target_columns[1]][start_idx:end_idx].plot(label=" LSTM", color='red', ax=ax2, alpha=0.7)
pred_series_gru_unscaled[target_columns[1]][start_idx:end_idx].plot(label=" GRU", color='orange', ax=ax2, alpha=0.7)
pred_series_tcn_unscaled[target_columns[1]][start_idx:end_idx].plot(label=" TCN", color='purple', alpha=0.7, title=f"Actual vs Predicted Energy Consumption {target_columns[1]}", ax=ax2)


<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise (Optional): Train Transformer/ARIMA/Prophet with DARTS</h2>

In this optional exercise, you'll explore additional forecasting approaches to complete your comparison:

### **Transformer Model**
1. Train a **Transformer model** using past covariates (weather data) (Attention: future covariates not supported)
2. Configure the attention mechanism with appropriate parameters:
   - `d_model`: Embedding dimension (64)
   - `nhead`: Number of attention heads (4) 
   - `num_encoder_layers` and `num_decoder_layers`: Network depth (2 each)

### **ARIMA Model (Classical Statistical)**
3. Train **ARIMA models** for time series forecasting
   - **Note**: ARIMA only supports univariate series, so train separate models for each target
   - Combine predictions from both models for evaluation

### **Prophet Model (Facebook's Forecasting Tool)**
4. Train a **Prophet model** with seasonality components:
   - Enable daily, weekly, and yearly seasonality
   - Use future covariates (time faeatures) to improve predictions

### **Evaluation & Comparison**
5. Compare all models using the same metrics (MAE, RMSE, MAPE, sMAPE)
6. Analyze training times and performance trade-offs
7. Visualize predictions to understand each model's behavior

**Learning Goals:**
- Understand when to use classical vs. modern approaches
- Compare attention mechanisms (Transformer) vs. convolutions (TCN) vs. recurrence (RNN/LSTM/GRU)
- See how statistical models (ARIMA/Prophet) perform against deep learning approaches
- Learn about multivariate forecasting strategies with univariate models

</div>

In [ ]:
from darts.models import TransformerModel

In [ ]:
# Transformer model
model_transformer = ...


⚠️ ARIMA is too slow (computation hungry). To run this resample the data to the "d" frequency (daily)

In [ ]:
from darts.models import ARIMA

start_time = time.time()

# ARIMA requires univariate series, so we train separate models for each target Model for Value_NE5 (Households)
print("Training ARIMA for Value_NE5 (Households)...")
train_ne5 = train_target["Value_NE5"]
test_ne5 = test_target["Value_NE5"]


model_arima_ne5 = ...
model_arima_ne5.fit(...)

pred_arima_ne5 = model_arima_ne5.historical_forecasts(
    ...
)

print("Training ARIMA for Value_NE7 (Business)...")
# Model for Value_NE7 (Business)

...

pred_arima_ne7=...

end_time = time.time()
training_time_arima = end_time - start_time

# Combine the two predictions into a multivariate series
pred_series_arima = pred_arima_ne5.stack(pred_arima_ne7)

# Inverse transform predictions
pred_series_arima_unscaled = ...
test_series_unscaled = ...

# Calculate evaluation metrics
evaluation_metrics_arima = {
    "MAE": ...,
    "RMSE": ...,
    "MAPE": ...,
    "sMAPE": ...,
}

In [ ]:
from darts.models import Prophet

In [ ]:
# Train the Prophet model (univariate only)

start_time = time.time()

# Prophet requires univariate series, so we train separate models for each target
# Model for Value_NE5 (Households)
train_ne5 = train_target["Value_NE5"]
test_ne5 = test_target["Value_NE5"]

model_prophet_ne5 = ...
model_prophet_ne5.fit(...)

pred_prophet_ne5 = model_prophet_ne5.historical_forecasts(
    ...
)

# Model for Value_NE7 (Business)
print("Training Prophet for Value_NE7 (Business)...")

...

pred_prophet_ne7 = ...

end_time = time.time()
training_time_prophet = end_time - start_time

# Combine the two predictions into a multivariate series
pred_series_prophet = pred_prophet_ne5.stack(pred_prophet_ne7)

# Inverse transform predictions
pred_series_prophet_unscaled = ...

# Calculate evaluation metrics
evaluation_metrics_prophet = {
    "MAE": ...,
    "RMSE": ...,
    "MAPE": ...,
    "sMAPE": ...,
}


In [ ]:
# Print comprehensive comparison of all models
print("=" * 60)
print("MODEL COMPARISON RESULTS")
print("=" * 60)

models_results = {
    "RNN": (evaluation_metrics_rnn, training_time_rnn),
    "LSTM": (evaluation_metrics_lstm, training_time_lstm),
    "GRU": (evaluation_metrics_gru, training_time_gru),
    "TCN": (evaluation_metrics_tcn, training_time_tcn),
    "Transformer": (evaluation_metrics_transformer, training_time_transformer),
    "ARIMA": (evaluation_metrics_arima, training_time_arima),
    "Prophet": (evaluation_metrics_prophet, training_time_prophet)
}

for model_name, (metrics, train_time) in models_results.items():
    print(f"\n{model_name}:")
    print(f"  sMAPE: {metrics['sMAPE']:.4f}")
    print(f"  MAE:   {metrics['MAE']:.4f}")
    print(f"  RMSE:  {metrics['RMSE']:.4f}")
    print(f"  Training Time: {train_time:.2f}s")
    print("-" * 40)  


## Discussion — Practical Takeaways
- **When to prefer TCN?** When parallelism matters and seasonal/local patterns over medium-to-long contexts dominate.
- **When to prefer Transformer?** When long-range, non-local dependencies are key (mind the memory cost of large context windows).
- **When to prefer GRU/LSTM?** With smaller datasets/models and moderate context lengths; often stable and efficient.
- **Hyperparameters mapping:** `window_size` ≈ context length. For TCN, the **receptive field** (kernel size, dilations, stacks) sets the effective context.
